In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('slide/database/features.db')
query = "SELECT channel, COUNT(well) as total_wells FROM LogsChannels GROUP BY channel ORDER BY total_wells DESC"
df = pd.read_sql_query(query, conn)

Analysis on the distribution of channels in the dataset.

The goal is to identify the most common channels and their frequency to improve data processing and analysis.

In [ ]:
import plotly.graph_objs as go
import plotly.offline as pyo

# Create bar trace for total_wells
bar_trace = go.Bar(
    x=df['channel'],
    y=df['total_wells'],
    name='Total Wells',
    marker=dict(color='rgba(31, 119, 180, 0.7)')
)

# Create line trace for cumulative percentage
line_trace = go.Scatter(
    x=df['channel'],
    y=df['cumperc'],
    name='Cumulative Percentage',
    yaxis='y2',
    mode='lines+markers',
    line=dict(color='orange', width=2)
)

layout = go.Layout(
    title='Interactive Pareto Chart of Channels by Total Wells',
    xaxis=dict(title='Channel', tickangle=45),
    yaxis=dict(title='Total Wells', showgrid=False),
    yaxis2=dict(
        title='Cumulative Percentage (%)',
        overlaying='y',
        side='right',
        range=[0, 110],
        showgrid=False
    ),
    legend=dict(x=0.75, y=1.15, orientation='h'),
    margin=dict(b=120),
    width=1000,
    height=600
)

fig_plotly = go.Figure(data=[bar_trace, line_trace], layout=layout)
pyo.iplot(fig_plotly)

In [4]:
import numpy as np

# Find which channels appear most often together for the same well

# Query all well-channel pairs
well_channel_df = pd.read_sql_query("SELECT well, channel FROM LogsChannels", conn)

# Create a pivot table: rows=well, columns=channel, values=1 if present
well_channel_matrix = pd.crosstab(well_channel_df['well'], well_channel_df['channel'])

# Compute the co-occurrence matrix (channels x channels)
co_occurrence = well_channel_matrix.T.dot(well_channel_matrix)

# Set diagonal to zero (ignore self-co-occurrence)
np.fill_diagonal(co_occurrence.values, 0)

# Find the top N most frequent channel pairs
co_occurrence_unstacked = co_occurrence.unstack()
co_occurrence_unstacked = co_occurrence_unstacked[co_occurrence_unstacked > 0]
co_occurrence_unstacked = co_occurrence_unstacked.sort_values(ascending=False)

# Display the top 10 most common channel pairs
co_occurrence_unstacked.head(10)

channel  channel
SP       GR         989
GR       SP         989
         CALI       890
CALI     GR         890
         SP         872
SP       CALI       872
GR       ILD        795
ILD      GR         795
DRHO     RHOB       778
RHOB     DRHO       778
dtype: int64

In [5]:
# Find the 8 channels that appear most often together (i.e., have the highest total co-occurrence with other channels)
# Sum co-occurrences for each channel (excluding self-co-occurrence)
channel_co_sum = co_occurrence.sum(axis=1)
top8_channels = channel_co_sum.sort_values(ascending=False).head(8)
top8_channels.index.tolist()

['GR', 'SP', 'CALI', 'RHOB', 'DRHO', 'NPHI', 'ILD', 'DT']

In [7]:
# Get the top 10 channels that co-occur most with SP
top10_with_SP = co_occurrence.loc['SP'].sort_values(ascending=False).head(10)
print("Top 10 channels that appear most often with SP:")
print(top10_with_SP)

# Get the top 10 channels that co-occur most with GR
top10_with_GR = co_occurrence.loc['GR'].sort_values(ascending=False).head(10)
print("\nTop 10 channels that appear most often with GR:")
print(top10_with_GR)

Top 10 channels that appear most often with SP:
channel
GR      989
CALI    872
ILD     772
DRHO    750
RHOB    750
DT      703
NPHI    663
MSFL    533
CILD    498
SFLU    479
Name: SP, dtype: int64

Top 10 channels that appear most often with GR:
channel
SP      989
CALI    890
ILD     795
RHOB    776
DRHO    775
DT      727
NPHI    688
MSFL    539
CILD    511
SFLU    484
Name: GR, dtype: int64


In [2]:
query = "SELECT * FROM Features;"
df_features = pd.read_sql_query(query, conn)

: 

In [ ]:
df_features = pd.read_sql_query(query, conn)
